# SFT training (Qwen2.5-0.5B)
Task 1 setup, data preparation for all three from-scratch runs, then SFT. Accelerator: GPU T4 x2, internet on.

## Setup
Set `GITHUB_USER` below. The repository is private, so add a Kaggle secret named `GITHUB_TOKEN` (a GitHub personal access token with read access) under *Add-ons → Secrets*. The token is passed as an HTTP header and is not written into the cloned repository.

In [ ]:
import base64
import os
import subprocess

from kaggle_secrets import UserSecretsClient

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "Small-Model_Alignment"
REPO_DIR = f"/kaggle/working/{REPO_NAME}"

if not os.path.isdir(REPO_DIR):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    credentials = base64.b64encode(f"{GITHUB_USER}:{token}".encode()).decode()
    clone_command = [
        "git",
        "-c",
        f"http.extraHeader=Authorization: Basic {credentials}",
        "clone",
        "-q",
        f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git",
        REPO_DIR,
    ]
    result = subprocess.run(clone_command)
    if result.returncode != 0:
        raise SystemExit("git clone failed")

os.chdir(REPO_DIR)
print(os.getcwd())

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers==4.57.6" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib "trl==1.16.0"
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi

In [ ]:
import datasets
import peft
import torch
import transformers
import trl

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("trl", trl.__version__)
print("peft", peft.__version__)
print("visible GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
if torch.cuda.device_count() < 2:
    raise SystemExit("Expected two GPUs. Select the GPU T4 x2 accelerator.")

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py

## Data preparation
Writes `outputs/data/<run>_train.jsonl`, `<run>_val.jsonl` and `<run>_data_stats.json` for `sft`, `dpo_hh` and `dpo_uf`. The DPO notebooks read this folder from this notebook's saved output.

In [ ]:
!python scripts/prepare_data.py --config configs/sft.yaml --output_dir outputs/data
!python scripts/prepare_data.py --config configs/dpo_hh.yaml --output_dir outputs/data
!python scripts/prepare_data.py --config configs/dpo_uf.yaml --output_dir outputs/data

## SFT
Single GPU (`cuda:0`), per-device batch 2 x accumulation 16 = effective batch 32. Validation loss every 100 optimizer steps and at the end; `best/` and `final/` are saved under `outputs/sft/`.

In [ ]:
!python scripts/train_sft.py --config configs/sft.yaml --data_dir outputs/data --output_dir outputs/sft

In [ ]:
!python scripts/plot_metrics.py --metrics_path outputs/sft/metrics.jsonl --output_dir outputs/sft/plots --run_name sft

## Archive small files
Checkpoints are not zipped. Use *Save Version* afterwards so `outputs/sft/best`, `outputs/sft/final` and `outputs/data` are kept in the notebook output.

In [ ]:
!cd outputs && zip -q -r /kaggle/working/sft_run_outputs.zip sft/metrics.jsonl sft/config_used.json sft/data_stats.json sft/prompt_template.txt sft/train_log.txt sft/plots data/sft_data_stats.json data/dpo_hh_data_stats.json data/dpo_uf_data_stats.json
!ls -lh /kaggle/working/sft_run_outputs.zip